In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,OrdinalEncoder
import warnings
warnings.filterwarnings(action='ignore')

In [ ]:
# datasets = ['tips', 'diamonds', 'fmri']
# dfs = [sns.load_dataset(name) for name in datasets]

# Data Preprocessing

In [ ]:
def compute_dtype(df):
    return df.dtypes
def compute_nunique(df):
    return df.nunique()
def compute_null_percentage(df):
    return (df.isnull().mean() * 100)

In [ ]:
def numerical_data(df):
    return df.select_dtypes(include=['number']).columns

In [ ]:
def categorical_data(df, threshold = 5):
    return [feature for feature in df.columns
          if df[feature].nunique()<=threshold ]

In [ ]:
def str_data(df):
    return df.select_dtypes(include=['object']).columns

In [ ]:
#Used for categorical variables.
def fill_na_mode(df):
    str_col=str_data(df)
    for feature in str_col:
        mode=df[feature].mode()[0]
        df[feature].fillna(mode, inplace=True)
    return df

In [ ]:
#Better for skewed distributions or data with outliers.
def fill_na_median(df):
    numerical_col=numerical_data(df)
    for feature in numerical_col:
        median=df[feature].median()
        df[feature].fillna(median, inplace=True)
    return df

In [ ]:
#Suitable for symmetric (normal) distributions without outliers.
def fill_na_mean(df):
    numerical_col=numerical_data(df)
    for feature in numerical_col:
        mean=df[feature].mean()
        df[feature].fillna(mean, inplace=True)
    return df

In [ ]:
def is_null(df):
    return df.isnull().sum()

In [ ]:
def drop_null_all_row(df):
    df.dropna(how='any',inplace=True)
    return df

In [ ]:
def drop_null_row(df,columns):
    df.dropna(subset=[columns],inplace=True)
    return df

In [ ]:
def is_duplicate(df):
    return df.duplicated().sum()

In [ ]:
def remove_duplicate(df):
    df.drop_duplicates(inplace = True) 

# EDA

In [ ]:
def plot_hist(df,x,xlabel,title,palette):
    plt.figure(figsize=(12, 2))
    sns.histplot(df[x], palette=palette)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel('Freq')
    plt.show()

In [ ]:
def plot_bar(df,x,y,title,palette):
    plt.figure(figsize=(12, 2))
    sns.barplot(x=x,y=y,data=df,palette=palette)
    plt.title(title)
    plt.show()

In [ ]:
def plot_pie(df,x,xlabel,ylabel,title,palette):
    plt.figure(figsize=(12, 2))
    plt.pie(df[x].unique(),labels=df[x].unique(),autopct="%1.1f%%", palette=palette)
    plt.title(title)
    plt.show()

In [ ]:
def plot_box(df,x,palette):
    plt.figure(figsize=(12, 2))
    sns.boxplot(data=df,x=x, palette=palette)
    plt.title('Box Plot')
    plt.show()

In [ ]:
def plot_scatter(x,y,z,xlabel,ylabel,palette):
    plt.figure(figsize=(12, 2))
    sns.scatterplot(x=x, y=y,hue=z, palette=palette)
    plt.title('Scatter Plot')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.show()

# Feature Engineering 

## 1.Numerical Feature Engineering 

### 1.1 Scalling

#### 1.1.1 Normalization 

##### Min-Max Scaling

In [ ]:
def mm_scaler(df):
    numerical_col=numerical_data(df)
    scaler=MinMaxScaler()
    for feature in numerical_col:
        df[feature]=scaler.fit_transform(df[[feature]])
    return df 

##### Log normalization

In [ ]:
def log_normal(df):
    numerical_col=numerical_data(df)
    for feature in numerical_col:
        negative_feature = df[feature] < 0
        df[feature] = np.log1p(np.abs(df[feature]))
        df[feature]=np.where(negative_feature,-df[feature],df[feature])
    return df 

#### 1.1.2 Standardization 

##### Z-Score Scaling

In [ ]:
def z_scaler(df):
    numerical_col=numerical_data(df)
    scaler=StandardScaler()
    for feature in numerical_col:
        df[feature]=scaler.fit_transform(df[[feature]])
    return df 

### 1.2 Discretization

In [ ]:
def bin_numerical_features(df, bins=10):
    numerical_col = numerical_data(df)
    for feature in numerical_col:
        df[feature + '_binned'] = pd.cut(df[feature], bins=bins, labels=False)
    return df

### 1.3 Descriptive Feature

#### 1.3.1 Skewness

In [ ]:
#Positive Skew-->(mean > median)
#Negative Skew-->(mean < median)
#Zero Skewness-->Symmetrical distribution (mean ≈ median) Normal distribution 

In [ ]:
#Log or Square Root Transformation (Best for Right Skew)
#Squaring (Best for Negative Skew)
def skewness(df):
    numerical_col=numerical_data(df)
    df_skew=df[numerical_col].apply(lambda x:x.skew())
    df_skew = df_skew.sort_values(ascending=False)
    df_skew=df_skew.reset_index()
    df_skew.columns = ['Feature', 'SkewFactor']
    return df_skew

#### 1.2.3 Quantile and Precentiles

In [ ]:
def quartiles(df):
    numerical_cols = numerical_data(df)
    return df[numerical_cols].quantile([0.25, 0.5, 0.75]).T

#### 1.3 Feature Transformation

In [ ]:
def sqrt_transform(df):
    numerical_col=numerical_data(df)
    for feature in numerical_col:
        df[feature] = np.sqrt(df[feature])
    return df 

In [ ]:
def sq_transform(df):
    numerical_col=numerical_data(df)
    for feature in numerical_col:
        df[feature] = np.square(df[feature])
    return df 

In [ ]:
def correlation(df,target):
    numerical_col=numerical_data(df)
    correlation_matrix = df[numerical_col].corr()
    correlation_target=correlation_matrix[target].sort_values(ascending=False)
    correlation_target=correlation_target.reset_index()
    correlation_target.columns=['feature','corr factor']
    return correlation_target

## 2.Categorical Feature Engineering

#### One-Hot Encoding.

In [ ]:
# def one_hot_encoding(df):
#     str_col=categorical_data(df)
#     encoder=OneHotEncoder( handle_unknown='ignore')
#     for feature in str_col:
#        df=encoder.fit_transform(df[[feature]])
#     return df
def one_hot_encoding(df):
    str_col=categorical_data(df)
    df=pd.get_dummies(df,columns=str_col)
    return df

#### Label Encoding

In [ ]:
def label_encoding(df):
    str_col=categorical_data(df)
    encoder=LabelEncoder()
    for feature in str_col:
        df[feature]=encoder.fit_transform(df[[feature]])
    return df

#### Ordinal Encoding

In [ ]:
def ordinal_encode(df,str_col):
    encoder = OrdinalEncoder()
    df[str_col] = encoder.fit_transform(df[[str_col]])
    return df

# Outliers Detection and Treatment

# Box Plot

![boxplot](boxplot.png)

In [ ]:
def box_plot(df):
    
    numerical_col=numerical_data(df)
    plt.figure()
    print('BoxPlot Before Handle Outliers')
    df[numerical_col].boxplot()

    for col in numerical_col:
        Q1=np.quantile(df[col],0.25)
        Q3=np.quantile(df[col],0.75)

        IQR = Q3 - Q1

        Lower_limit=Q1-1.5*IQR
        Upper_limit=Q1+1.5*IQR

        df[col]=np.where(df[col]<Lower_limit,Lower_limit,df[col])
        df[col]=np.where(df[col]>Upper_limit,Upper_limit,df[col])
    
    print('BoxPlot After Handle Outliers')
    df[numerical_col].boxplot()

# Z-Score Detect

# Isolation Forest

In [ ]:
from sklearn.tree import 

# KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier,kneighbors_graph


In [ ]:
data = {
    'A': [1, 2, 3, 4, 5, 100],
    'B': [10, 20, 30, 40, 50, 200],
    'C': ['X', 'Y', 'Z', 'X', 'Y', 'Z']  # Non-numeric column
}
df=pd.DataFrame(data)

In [ ]:
box_plot(df)